In [1]:
import os, sys
print(sys.executable)
print(os.getcwd())

c:\Users\Ryan\Documents\Berkeley_MFE\Pre_Program_Courses\Python\UCB-MFE-python-preprogram-2026-winter\.venv\Scripts\python.exe
c:\Users\Ryan\Documents\Berkeley_MFE\Pre_Program_Courses\Python\UCB-MFE-python-preprogram-2026-winter\Homeworks\HW2\Ryan_Beauchamp


In [2]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "data/data.db"
conn = sqlite3.connect(DB_PATH)

raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)  # never modify this
ref = pd.read_sql_query("SELECT * FROM ohlc", conn)     # reference calendar / adjustment factors
print(raw.shape, ref.shape)

(397711, 8) (368251, 8)


In [3]:
raw.info()
display(raw.head())
display(raw.describe(include='all').T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ts,397711,15868,2020-11-24,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
open,397711.0,NaN,NaN,NaN,46.285479,67.132594,0.0,4.209938,19.85,57.53,604.26
high,397711.0,NaN,NaN,NaN,46.74766,67.77821,0.005208,4.24,20.06,58.13,608.63
low,397711.0,NaN,NaN,NaN,45.816279,66.477178,0.004801,4.16,19.63,56.95,597.88
close,397711.0,NaN,NaN,NaN,46.292903,67.141879,0.005208,4.2,19.86,57.53,605.4
adj_close,377856.0,NaN,NaN,NaN,51.65225,194.584045,0.002945,1.39,11.96,43.77,10442.510783
volume,397711.0,NaN,NaN,NaN,30977733.689946,126623650.175253,0.0,1730700.0,4648200.0,11023796.5,9276606399.0
ticker,397711,150,PG,15450,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("Raw ticker strings:", raw["ticker"].nunique())
print(raw["ticker"].drop_duplicates().head(20).tolist())
print("ts dtype:", raw["ts"].dtype, "| sample:", raw["ts"].iloc[0:3].tolist())
print("ts min/max (as text):", raw["ts"].min(), raw["ts"].max())
print("Nulls per column:")
display(raw.isna().sum())
num_cols = ["open", "high", "low", "close", "adj_close", "volume"]
print("Non-finite per column:")
display((~np.isfinite(raw[num_cols].astype(float))).sum() - raw[num_cols].isna().sum())

Raw ticker strings: 150
['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ', 'DIS', '<a target="_blank"><span style="color: blue">CSCO</span></a>', 'SHW', 'KO', '<a target="_blank"><span style="font-weight: bold; color: blue">HD</span></a>', '<a target="_blank"><span style="font-weight: bold; color: blue">VZ</span></a>', 'IBM', 'PG', 'HON', 'AXP', 'MMM', 'GS', 'BA', 'CRM']
ts dtype: object | sample: ['1975-04-14', '1978-04-05', '2018-05-25']
ts min/max (as text): 1962-01-02 2024-12-06
Nulls per column:


ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64

Non-finite per column:


open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

In [5]:
# Ticker normalization (on a copy so that the raw remains untouched)
ticker_clean = (raw["ticker"].astype("string")
                .str.replace(r"<[^>]+>", "", regex=True)
                .str.strip().str.upper())
print("Clean tickers:", ticker_clean.nunique())
print(sorted(ticker_clean.unique()))

tmp = raw.assign(ticker_clean=ticker_clean)
display(tmp.groupby("ticker_clean")["ts"].agg(first="min", last="max", rows="size"))

Clean tickers: 30
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


,first,last,rows
ticker_clean,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


In [6]:
# Duplicates: exact vs. conflicting keys
cols = ["ticker_clean", "ts", "open", "high", "low", "close", "adj_close", "volume"]
print("Exact duplicate rows:", tmp.duplicated(cols).sum())
de_duplicate = tmp.drop_duplicates(cols)
print("Conflicting (ticker, ts) keys after exact de-duplication:", de_duplicate.duplicated(["ticker_clean", "ts"]).sum())

Exact duplicate rows: 29460
Conflicting (ticker, ts) keys after exact de-duplication: 0


In [7]:
# Quick look at the suspicious spots
print("open <= 0:", (raw["open"] <= 0).sum(), "| zero volume:", (raw["volume"] == 0).sum())
display((raw["adj_close"] / raw["close"]).describe())

open <= 0: 5 | zero volume: 81


count    377856.000000
mean          0.848215
std           1.638156
min           0.101742
25%           0.421354
50%           0.634915
75%           0.833722
max          23.973383
dtype: float64

In [8]:
# Address the five rows with an open below 0
cols_show = ["ticker_clean", "ts", "open", "high", "low", "close", "adj_close", "volume"]
display(tmp.loc[tmp["open"] <= 0, cols_show])

,ticker_clean,ts,open,high,low,close,adj_close,volume
99637,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000
224790,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800
287343,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840
350808,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0
364568,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830


In [9]:
# Take a look at when the Adj Close is notably larger than the close
tmp["adj_ratio"] = tmp['adj_close'] / tmp["close"]
display(tmp.loc[tmp["adj_ratio"] > 1].groupby("ticker_clean").size().sort_values(ascending=False).head(10))
display(tmp.loc[tmp['adj_ratio'] > 5, cols_show + ["adj_ratio"]].head(10))

ticker_clean
BA     906
MMM    538
IBM    506
CAT    491
KO     484
MCD    478
JNJ    478
MRK    478
HON    472
CVX    457
dtype: int64

,ticker_clean,ts,open,high,low,close,adj_close,volume,adj_ratio
66,CSCO,1992-07-30,0.72223,0.73265,0.71876,0.73091,5.681972,29649600,7.773833
113,BA,2013-08-26,105.75000,106.52000,105.21000,105.53000,1584.835541,2482146,15.017867
122,HON,2021-12-29,207.91000,208.04000,206.46000,207.53000,1085.759511,1714622,5.231820
174,TRV,1987-02-18,12.88000,13.13000,12.81000,13.00000,104.130431,1281600,8.010033
269,HD,1991-08-21,5.64000,5.90000,5.64000,5.81000,39.879497,12040910,6.863941
282,HD,1990-09-18,2.46000,2.49000,2.41000,2.49000,33.380542,5124600,13.405840
291,IBM,1988-01-06,29.10000,29.43000,28.95000,29.34000,206.347935,6604520,7.032990
292,AXP,2022-10-20,144.00000,147.32000,142.15000,142.42000,2149.535814,4319336,15.092935
335,CAT,1991-12-31,5.34000,5.50000,5.30000,5.48000,51.352279,1780000,9.370854
356,WMT,1985-03-07,0.47005,0.47005,0.46224,0.46224,3.731076,24710424,8.071729


In [10]:
# Compare against the reference table
print (ref.columns.tolist(), "| ref tickers:", ref["ticker"].nunique())
m = de_duplicate.merge(ref, left_on=["ticker_clean", "ts"], right_on=["ticker", "ts"], how="left", suffixes=("_hw", "_ref"), indicator=True)

print(m["_merge"].value_counts())
for c in ["open", "high", "low", "close", "adj_close", "volume"]:
    differs = ~np.isclose(m[f"{c}_hw"], m[f"{c}_ref"], equal_nan=True)
    print(f"{c}: {differs.sum()} rows differ from the reference dataframe")

['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker'] | ref tickers: 30
_merge
both          368251
left_only          0
right_only         0
Name: count, dtype: int64
open: 0 rows differ from the reference dataframe
high: 0 rows differ from the reference dataframe
low: 0 rows differ from the reference dataframe
close: 0 rows differ from the reference dataframe
adj_close: 28791 rows differ from the reference dataframe
volume: 0 rows differ from the reference dataframe


In [11]:
hw_null = m["adj_close_hw"].isna()
diff = ~np.isclose(m["adj_close_hw"], m["adj_close_ref"], equal_nan=True)
print("Differs and hw is missing: ", (diff & hw_null).sum())
print("Differs and hw has a value: ", (diff & ~hw_null).sum())
print ("Nulls in ref adj_close: ", m["adj_close_ref"].isna().sum())

bad = m[diff & ~hw_null].copy()
bad["hw_over_ref"] = bad["adj_close_hw"] / bad["adj_close_ref"]
display(bad["hw_over_ref"].describe())
display(bad[["ticker_clean", "ts", "close_hw", "adj_close_hw", "adj_close_ref", "hw_over_ref"]].head(10))
display(bad.groupby("ticker_clean").size().sort_values(ascending=False).head())

print("ref adj/close ratio:")
display((m["adj_close_ref"] / m["close_ref"]).describe())

Differs and hw is missing:  18374
Differs and hw has a value:  10417
Nulls in ref adj_close:  0


count    10417.000000
mean        14.081361
std          5.761729
min          4.002156
25%          9.185794
50%         14.064218
75%         19.060007
max         23.998439
Name: hw_over_ref, dtype: float64

,ticker_clean,ts,close_hw,adj_close_hw,adj_close_ref,hw_over_ref
66,CSCO,1992-07-30,0.73091,5.681972,0.487720,11.650071
113,BA,2013-08-26,105.53000,1584.835541,89.865974,17.635546
122,HON,2021-12-29,207.53000,1085.759511,195.730000,5.547231
174,TRV,1987-02-18,13.00000,104.130431,4.380000,23.774071
269,HD,1991-08-21,5.81000,39.879497,3.390000,11.763863
278,HON,1993-03-29,15.95000,72.491131,8.230000,8.808157
282,HD,1990-09-18,2.49000,33.380542,1.450000,23.021064
291,IBM,1988-01-06,29.34000,206.347935,11.310000,18.244733
292,AXP,2022-10-20,142.42000,2149.535814,138.680000,15.499970
335,CAT,1991-12-31,5.48000,51.352279,2.520000,20.377888


ticker_clean
MMM    499
IBM    476
BA     465
CAT    459
KO     452
dtype: int64

ref adj/close ratio:


count    368251.000000
mean          0.610928
std           0.247348
min           0.101742
25%           0.411778
50%           0.622214
75%           0.809410
max           1.000000
dtype: float64

In [12]:
wrong = diff & ~hw_null
rule_gt1 = m["adj_close_hw"] > m["close_hw"]    # NaN compares as False
print("Wrong rows caught by adj > close:", (wrong & rule_gt1).sum(), "of", wrong.sum())
print("Flagged by the rule but not wrong:", (~wrong & ~hw_null & rule_gt1).sum())

Wrong rows caught by adj > close: 10348 of 10417
Flagged by the rule but not wrong: 380


In [13]:
fp = ~wrong & ~hw_null & rule_gt1
print("False-positive ratio, adj/close:")
display((m.loc[fp, "adj_close_hw"] / m.loc[fp, "close_hw"]).describe())

miss = wrong & ~rule_gt1
print("Missed rows, TRUE ratio (ref adj / close):")
display((m.loc[miss, "adj_close_ref"] / m.loc[miss, "close_hw"]).describe())

False-positive ratio, adj/close:


count    3.800000e+02
mean     1.000000e+00
std      1.128434e-08
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

Missed rows, TRUE ratio (ref adj / close):


count    69.000000
mean      0.146696
std       0.038452
min       0.106295
25%       0.119703
50%       0.125404
75%       0.179949
max       0.234095
dtype: float64

In [14]:
# Factor spike detector: today's adj/close factor vs. the ticker's local median
s = m.sort_values(["ticker_clean", "ts"])              # must be time-ordered within ticker
factor = s["adj_close_hw"] / s["close_hw"]
med = factor.groupby(s["ticker_clean"]).transform(
    lambda x: x.rolling(11, center=True, min_periods=3).median())
m["factor_spike"] = (factor / med).reindex(m.index)    # put results back in m's original order

print("Spike ratio, WRONG rows:")
display(m.loc[wrong, "factor_spike"].describe())
print("Spike ratio, GOOD rows with a value:")
display(m.loc[~wrong & ~hw_null, "factor_spike"].describe())

Spike ratio, WRONG rows:


count    10417.000000
mean        14.078686
std          5.761159
min          3.976161
25%          9.185447
50%         14.062283
75%         19.063532
max         24.036769
Name: factor_spike, dtype: float64

Spike ratio, GOOD rows with a value:


count    339460.000000
mean          0.999967
std           0.001897
min           0.795067
25%           0.999939
50%           1.000000
75%           1.000045
max           1.523269
Name: factor_spike, dtype: float64

In [15]:
SPIKE_THRESHOLD = 2.5
TOL = 1e-6

flag_gt = m["adj_close_hw"] > m["close_hw"] * (1 + TOL)
flag_spike = m["factor_spike"] > SPIKE_THRESHOLD
flag_adj_invalid = flag_gt | flag_spike

print("Flagged by adj > close:   ", flag_gt.sum())
print("Flagged by spike:         ", flag_spike.sum())
print("Flagged by either:        ", flag_adj_invalid.sum())
print("Truly wrong (vs ref):     ", wrong.sum())
print("Missed (wrong, not flagged):  ", (wrong & ~flag_adj_invalid).sum())
print("False alarms (flagged, not wrong):", (flag_adj_invalid & ~wrong).sum())

Flagged by adj > close:    10348
Flagged by spike:          10417
Flagged by either:         10417
Truly wrong (vs ref):      10417
Missed (wrong, not flagged):   0
False alarms (flagged, not wrong): 0


In [16]:
def make_issues(df, mask, issue_type, severity, description):
    """Turn a True/False mask into quality-report rows."""
    out = df.loc[mask, ["ticker_clean", "ts"]].rename(
        columns={"ticker_clean": "ticker", "ts": "issue_date"})
    out["issue_type"] = issue_type
    out["description"] = description
    out["severity"] = severity
    return out[["ticker", "issue_type", "issue_date", "description", "severity"]]

In [17]:
b = m.rename(columns={f"{c}_hw": c for c in ["open", "high", "low", "close", "adj_close", "volume"]})
issues = []

# 1. Missing and invalid adj_close (already diagnosed)
issues.append(make_issues(b, b["adj_close"].isna(), "missing_adj_close", "medium",
                          "adj_close is null; raw OHLC present"))
issues.append(make_issues(b, flag_adj_invalid, "invalid_adj_close", "high",
                          "adj_close/close factor spikes >2.5x local median, or adj_close > close"))

# 2. Nonpositive or non-finite prices
for c in ["open", "high", "low", "close"]:
    bad = b[c].notna() & ~(np.isfinite(b[c]) & (b[c] > 0))
    issues.append(make_issues(b, bad, "nonpositive_or_nonfinite_price", "high", f"{c} <= 0 or not finite"))

# 3. Volume
issues.append(make_issues(b, b["volume"] == 0, "zero_volume", "low", "zero volume; possible stale data"))
bad_vol = b["volume"].notna() & ~(np.isfinite(b["volume"]) & (b["volume"] >= 0))
issues.append(make_issues(b, bad_vol, "invalid_volume", "high", "volume negative or not finite"))

# 4. OHLC bound violations
bounds = (b["low"] > b["open"]) | (b["open"] > b["high"]) | (b["low"] > b["close"]) | (b["close"] > b["high"])
issues.append(make_issues(b, bounds, "ohlc_bounds_violation", "high", "violates low <= open/close <= high"))

# 5. Exact duplicates (on the pre-dedup table)
key_cols = ["ticker_clean", "ts", "open", "high", "low", "close", "adj_close", "volume"]
issues.append(make_issues(tmp, tmp.duplicated(key_cols), "duplicate_record", "low",
                          "exact duplicate row; dropped"))

# 6. 3-sigma return outliers (simple returns on raw close, per ticker, sample std)
s = b.sort_values(["ticker_clean", "ts"])
ret = s.groupby("ticker_clean")["close"].pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
mu = ret.groupby(s["ticker_clean"]).transform("mean")
sd = ret.groupby(s["ticker_clean"]).transform(lambda x: x.std(ddof=1))
outlier = ((ret - mu).abs() > 3 * sd) & (sd > 0)
issues.append(make_issues(s, outlier, "return_outlier_3sigma", "low", "|return - mean| > 3 sample std; review, not an error"))

# 7. Missing trading dates, using the reference calendar
def find_missing_dates(obs, ref_df):
    o = obs[["ticker_clean", "ts"]].drop_duplicates()
    r = ref_df[["ticker", "ts"]].rename(columns={"ticker": "ticker_clean"}).drop_duplicates()
    j = r.merge(o, on=["ticker_clean", "ts"], how="left", indicator=True)
    return j.loc[j["_merge"] == "left_only", ["ticker_clean", "ts"]]

gaps = find_missing_dates(de_duplicate, ref)
print("Missing dates in real data:", len(gaps))
test_gap = find_missing_dates(de_duplicate.drop(index=de_duplicate.index[1000]), ref)
print("Detector test (one date removed):"); display(test_gap)
if len(gaps):
    issues.append(make_issues(gaps, pd.Series(True, index=gaps.index), "missing_date", "high",
                              "date in reference calendar absent from ohlc_hw"))

quality_report = pd.concat(issues, ignore_index=True)
display(quality_report.groupby(["issue_type", "severity"]).size())

Missing dates in real data: 0
Detector test (one date removed):


,ticker_clean,ts
321982,AAPL,2002-01-07


issue_type                      severity
duplicate_record                low         29460
invalid_adj_close               high        10417
missing_adj_close               medium      18374
nonpositive_or_nonfinite_price  high            5
ohlc_bounds_violation           high            5
return_outlier_3sigma           low          5056
zero_volume                     low            75
dtype: int64

In [18]:
clean = b.copy()   # de-duplicated, normalized tickers, with the _ref columns attached

# 1. Rebuild flagged adj_close from the reference factor
clean["ref_factor"] = clean["adj_close_ref"] / clean["close_ref"]
need_fix = clean["adj_close"].isna() | flag_adj_invalid
clean["adj_close_repaired"] = need_fix
clean.loc[need_fix, "adj_close"] = clean.loc[need_fix, "close"] * clean.loc[need_fix, "ref_factor"]
print("adj_close rows repaired:", need_fix.sum())

# 2. Quarantine nonpositive opens (log them, then remove)
quarantine_mask = (clean["open"] <= 0) | (clean["low"] > clean["open"]) | (clean["open"] > clean["high"])
quarantined = clean[quarantine_mask].copy()
display(quarantined[["ticker_clean", "ts", "open", "high", "low", "close"]])
clean = clean[~quarantine_mask].copy()

adj_close rows repaired: 28791


,ticker_clean,ts,open,high,low,close
97792,PG,1962-01-02,0.0,1.44000,1.42000,1.43000
215385,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955
271962,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000
327908,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338
339808,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537


In [19]:
# 3. Validate: these asserts should all pass silently
px = ["open", "high", "low", "close", "adj_close"]
assert np.isfinite(clean[px]).all().all() and (clean[px] > 0).all().all()
assert (clean["low"] <= clean["open"]).all() and (clean["open"] <= clean["high"]).all()
assert (clean["low"] <= clean["close"]).all() and (clean["close"] <= clean["high"]).all()
assert (clean["adj_close"] <= clean["close"] * (1 + 1e-6)).all()
assert not clean.duplicated(["ticker_clean", "ts"]).any()
print("All validation checks passed. Rows:", len(clean))

All validation checks passed. Rows: 368246


In [20]:
# Flags and return column, aligned to clean's current index (before any re-sorting)
clean["volume_repaired"] = clean["volume"].isna()          # none expected; the flag is kept for the schema
clean["zero_volume_flag"] = clean["volume"] == 0
clean["simple_return"] = ret.reindex(clean.index)
clean["return_outlier_flag"] = outlier.reindex(clean.index).fillna(False)

# Proof: unflagged vendor adj_close values equal the reference
untouched = ~clean["adj_close_repaired"]
print("Unflagged rows:", untouched.sum(),
      "| all equal to reference:", np.isclose(clean.loc[untouched, "adj_close"],
                                               clean.loc[untouched, "adj_close_ref"]).all())

# Outlier review: the 10 largest flagged moves
top = clean[clean["return_outlier_flag"]].assign(abs_ret=lambda d: d["simple_return"].abs()).nlargest(10, "abs_ret")
display(top[["ticker_clean", "ts", "close", "simple_return"]])

Unflagged rows: 339455 | all equal to reference: True


,ticker_clean,ts,close,simple_return
142198,TRV,1981-08-10,12.060000,1.000000
356588,TRV,1976-10-28,9.220000,1.000000
8968,TRV,1978-12-20,8.030000,0.997512
352553,MCD,1968-01-02,0.174900,-0.753620
107932,AAPL,2000-09-29,0.459820,-0.518695
144261,TRV,1981-08-11,6.030000,-0.500000
259619,TRV,1976-10-29,4.690000,-0.491323
259652,TRV,1978-12-21,4.090000,-0.490660
233696,NVDA,2000-03-07,0.173570,0.424106
358638,NVDA,2004-08-06,0.078584,-0.352314


In [21]:
def repair_prices(df):
    df = df.sort_values(["ticker", "ts"]).copy()
    px = ["open", "high", "low", "close"]
    df = df[~df[px].isna().all(axis=1)].copy()                 # 1. drop rows with ALL prices missing, before filling
    bad = (df["close"] > df["high"]) | (df["close"] < df["low"])
    df.loc[bad, "close"] = np.nan                              # 2. impossible close -> NaN (Option 1)
    df["price_repaired"] = df[px].isna().any(axis=1)
    df[px] = df.groupby("ticker")[px].ffill()                  # 3. fill within each ticker only
    df["volume_repaired"] = df["volume"].isna()                # 4. unknown volume -> 0, with a flag
    df["volume"] = df["volume"].fillna(0)
    return df

toy = pd.DataFrame({
    "ticker": ["A", "A", "A", "A", "B", "B"],
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-02", "2024-01-03"],
    "open":  [10, 10.5, np.nan, 11, np.nan, 20],
    "high":  [11, 11.0, np.nan, 12, 21, 21],
    "low":   [9, 10.0, np.nan, 10, 19, 19],
    "close": [10.5, 12.0, np.nan, 11.5, np.nan, 20.5],   # A day 2: close > high. B day 1: leading gap.
    "volume": [100, 200, 300, np.nan, 400, 500],
})
display(toy)
fixed = repair_prices(toy)
display(fixed)
# Rows still NaN after the fill are leading gaps. They are not extrapolated, so quarantine them.
print("Unresolved (quarantine):"); display(fixed[fixed[["open","high","low","close"]].isna().any(axis=1)])

,ticker,ts,open,high,low,close,volume
0,A,2024-01-02,10.0,11.0,9.0,10.5,100.0
1,A,2024-01-03,10.5,11.0,10.0,12.0,200.0
2,A,2024-01-04,NaN,NaN,NaN,NaN,300.0
3,A,2024-01-05,11.0,12.0,10.0,11.5,NaN
4,B,2024-01-02,NaN,21.0,19.0,NaN,400.0
5,B,2024-01-03,20.0,21.0,19.0,20.5,500.0


,ticker,ts,open,high,low,close,volume,price_repaired,volume_repaired
0,A,2024-01-02,10.0,11.0,9.0,10.5,100.0,False,False
1,A,2024-01-03,10.5,11.0,10.0,10.5,200.0,True,False
3,A,2024-01-05,11.0,12.0,10.0,11.5,0.0,False,True
4,B,2024-01-02,NaN,21.0,19.0,NaN,400.0,True,False
5,B,2024-01-03,20.0,21.0,19.0,20.5,500.0,False,False


Unresolved (quarantine):


,ticker,ts,open,high,low,close,volume,price_repaired,volume_repaired
4,B,2024-01-02,NaN,21.0,19.0,NaN,400.0,True,False


In [22]:
cols = ["open", "high", "low", "close", "adj_close", "volume"]
display(pd.concat({"before (raw)": raw[cols].describe().T,
                   "after (clean)": clean[cols].describe().T}, axis=1).round(3))

print("Raw rows:               ", len(raw))
print("Exact duplicates dropped:", 29460)
print("Quarantined (open = 0):  ", len(quarantined))
print("Retained:                ", len(clean))

out = (clean.rename(columns={"ticker_clean": "ticker"})
            [["ticker", "ts", "open", "high", "low", "close", "adj_close", "volume",
              "adj_close_repaired", "volume_repaired", "zero_volume_flag", "return_outlier_flag"]]
            .sort_values(["ticker", "ts"]).reset_index(drop=True))
out["ts"] = pd.to_datetime(out["ts"]).dt.strftime("%Y-%m-%d")
out.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)

print(pd.read_sql_query("SELECT COUNT(*) AS n, COUNT(DISTINCT ticker) AS tickers, MIN(ts) AS first, MAX(ts) AS last FROM ohlc_hw_cleaned", conn))

before (raw)                                                 \
                 count          mean           std    min         25%   
open          397711.0  4.628500e+01  6.713300e+01  0.000        4.21   
high          397711.0  4.674800e+01  6.777800e+01  0.005        4.24   
low           397711.0  4.581600e+01  6.647700e+01  0.005        4.16   
close         397711.0  4.629300e+01  6.714200e+01  0.005        4.20   
adj_close     377856.0  5.165200e+01  1.945840e+02  0.003        1.39   
volume        397711.0  3.097773e+07  1.266237e+08  0.000  1730700.00   

                                                 after (clean)                \
                  50%          75%           max         count          mean   
open            19.85        57.53  6.042600e+02      368246.0  4.630900e+01   
high            20.06        58.13  6.086300e+02      368246.0  4.677200e+01   
low             19.63        56.95  5.978800e+02      368246.0  4.584000e+01   
close           19.86        57.53  6.054000e+02      368246.0  4.631700e+01   
adj_close       11.96        43.77  1.044251e+04      368246.0  3.727500e+01   
volume     4648200.00  11023796.50  9.276606e+09      368246.0  3.095184e+07   

                                                                      \
                    std    min         25%         50%           75%   
open       6.714300e+01  0.005        4.21       19.89  5.756000e+01   
high       6.778900e+01  0.005        4.25       20.09  5.816000e+01   
low        6.648900e+01  0.005        4.17       19.67  5.699800e+01   
close      6.715500e+01  0.005        4.21       19.91  5.757000e+01   
adj_close  6.358800e+01  0.003        1.31       11.46  4.157000e+01   
volume     1.266153e+08  0.000  1731514.50  4649648.00  1.102160e+07   

                         
                    max  
open       6.042600e+02  
high       6.086300e+02  
low        5.978800e+02  
close      6.054000e+02  
adj_close  6.054000e+02  
volume     9.276606e+09

Raw rows:                397711
Exact duplicates dropped: 29460
Quarantined (open = 0):   5
Retained:                 368246
        n  tickers       first        last
0  368246       30  1962-01-02  2024-12-06


In [23]:
# Reversal test: does a flagged move undo itself the next day? (spike-and-revert = error signature)
c = clean.sort_values(["ticker_clean", "ts"]).copy()
c["next_ret"] = c.groupby("ticker_clean")["simple_return"].shift(-1)
c["round_trip"] = (1 + c["simple_return"]) * (1 + c["next_ret"])     # ~1.0 means the move reversed
flagged = c[c["return_outlier_flag"]]
rev = flagged[(flagged["round_trip"] - 1).abs() < 0.05]
print("Flagged outliers:", len(flagged), "| reversed next day (within 5%):", len(rev))
display(rev.groupby("ticker_clean").size().sort_values(ascending=False))

# The bars around the three TRV events
w = c[c["ticker_clean"] == "TRV"]
for d in ["1981-08-10", "1976-10-28", "1978-12-20"]:
    pos = w.index.get_loc(w.index[w["ts"] == d][0])
    display(w.iloc[pos-2:pos+3][["ts", "open", "high", "low", "close", "volume", "simple_return"]])

Flagged outliers: 5056 | reversed next day (within 5%): 1407


ticker_clean
KO      103
MMM      95
PG       94
IBM      93
JNJ      79
MRK      75
VZ       69
CVX      64
HON      58
JPM      54
AXP      53
DIS      51
CAT      51
BA       50
SHW      50
MCD      48
AMGN     41
WMT      40
MSFT     38
HD       31
CSCO     28
UNH      28
NKE      25
V        20
GS       17
AAPL     15
TRV      13
AMZN     10
CRM      10
NVDA      4
dtype: int64

,ts,open,high,low,close,volume,simple_return
257648,1981-08-06,6.06,6.06,6.06,6.06,60000,-0.003289
330542,1981-08-07,6.03,6.03,6.03,6.03,264800,-0.004950
142198,1981-08-10,12.06,12.06,12.06,12.06,0,1.000000
144261,1981-08-11,6.03,6.03,6.03,6.03,24800,-0.500000
26518,1981-08-12,6.13,6.13,6.13,6.13,267200,0.016584


,ts,open,high,low,close,volume,simple_return
71963,1976-10-26,4.50,4.50,4.50,4.50,91200,-0.004425
151849,1976-10-27,4.61,4.61,4.61,4.61,230400,0.024444
356588,1976-10-28,9.22,9.22,9.22,9.22,0,1.000000
259619,1976-10-29,4.69,4.69,4.69,4.69,79200,-0.491323
250542,1976-11-01,4.66,4.66,4.66,4.66,59200,-0.006397


,ts,open,high,low,close,volume,simple_return
291923,1978-12-18,4.02,4.02,4.02,4.02,155200,-0.040573
352720,1978-12-19,4.02,4.02,4.02,4.02,72000,0.000000
8968,1978-12-20,8.03,8.03,8.03,8.03,0,0.997512
259652,1978-12-21,4.09,4.09,4.09,4.09,105600,-0.490660
340063,1978-12-22,4.09,4.09,4.09,4.09,91200,0.000000


In [24]:
cand = c[c["return_outlier_flag"] & (c["volume"] == 0)
         & ((c["round_trip"] - 1).abs() < 0.02) & (c["simple_return"].abs() > 0.10)]
print("Spike candidates (big move + tight reversal + zero volume):", len(cand))
display(cand.groupby("ticker_clean").size())
display(cand[["ticker_clean", "ts", "close", "simple_return", "next_ret", "round_trip"]].head(15))

print("Zero-volume rows total:", (c["volume"] == 0).sum())
big_rev = c[c["return_outlier_flag"] & (c["simple_return"].abs() > 0.10)
            & ((c["round_trip"] - 1).abs() < 0.02) & (c["volume"] > 0)]
print("Big tight reversals WITH volume > 0:", len(big_rev))
display(big_rev[["ticker_clean", "ts", "close", "volume", "simple_return", "next_ret"]].head(10))

Spike candidates (big move + tight reversal + zero volume): 4


ticker_clean
AXP    1
TRV    3
dtype: int64

,ticker_clean,ts,close,simple_return,next_ret,round_trip
232637,AXP,1972-12-14,4.01,-0.126362,0.147132,1.002179
356588,TRV,1976-10-28,9.22,1.000000,-0.491323,1.017354
8968,TRV,1978-12-20,8.03,0.997512,-0.490660,1.017413
142198,TRV,1981-08-10,12.06,1.000000,-0.500000,1.000000


Zero-volume rows total: 74
Big tight reversals WITH volume > 0: 26


,ticker_clean,ts,close,volume,simple_return,next_ret
285620,AAPL,1997-08-07,0.26060,3707572856,0.109267,-0.081351
157599,AMZN,1997-09-12,0.18438,66672000,0.168367,-0.161026
2280,AMZN,1997-10-27,0.21354,121824000,-0.150259,0.158565
173399,AMZN,1998-07-06,1.16000,937668000,0.126214,-0.120690
104703,AMZN,1999-01-11,4.62000,778460000,0.152120,-0.116883
87083,AMZN,2001-01-24,1.09000,276962000,0.150202,-0.128440
288424,AXP,1975-08-04,2.40000,1553,-0.133574,0.145833
75992,BA,1997-10-27,43.00000,11777200,-0.112258,0.116279
356316,CSCO,2002-10-15,10.99000,130921872,0.100100,-0.107370
284266,CSCO,2020-03-12,33.20000,51855305,-0.103914,0.133735


In [25]:
big = c[c["return_outlier_flag"] & (c["simple_return"].abs() > 0.10)]
print(big.groupby("ts").size().sort_values(ascending=False).head(10))

ts
1987-10-19    23
2020-03-24    19
2008-10-13    18
2008-10-28    17
2020-03-16    17
1987-10-21    12
2008-09-29    10
2020-03-12    10
2020-03-13    10
1987-10-20    10
dtype: int64


In [ ]:
c[(c["ticker_clean"] == "AXP") & c["ts"].between("1972-12-12", "1972-12-18")][["ts","open","high","low","close","volume","simple_return"]]

,ts,open,high,low,close,volume,simple_return
130400,1972-12-12,4.63,4.63,4.63,4.63,672324,-0.006438
218769,1972-12-13,4.59,4.59,4.59,4.59,844816,-0.008639
232637,1972-12-14,4.01,4.01,4.01,4.01,0,-0.126362
27166,1972-12-15,4.60,4.60,4.60,4.60,979282,0.147132
52349,1972-12-18,4.60,4.60,4.60,4.60,421049,0.000000


In [27]:
fix_cols = ["open", "high", "low", "close"]
spike_idx = cand.index
before = clean.loc[spike_idx, ["ticker_clean", "ts"] + fix_cols + ["adj_close"]].copy()

# 1. Interpolate each spike bar from valid neighbors, within ticker, no extrapolation
t2 = c.copy()
t2.loc[spike_idx, fix_cols] = np.nan
t2[fix_cols] = t2.groupby("ticker_clean")[fix_cols].transform(
    lambda x: x.interpolate(method="linear", limit_area="inside"))
clean.loc[spike_idx, fix_cols] = t2.loc[spike_idx, fix_cols]

# 2. Recompute adj_close for those rows from the repaired close and the reference factor
clean.loc[spike_idx, "adj_close"] = clean.loc[spike_idx, "close"] * clean.loc[spike_idx, "ref_factor"]
clean["spike_repaired"] = False
clean.loc[spike_idx, "spike_repaired"] = True

# 3. Recompute returns and outlier flags on the repaired closes
clean = clean.sort_values(["ticker_clean", "ts"]).copy()
r = clean.groupby("ticker_clean")["close"].pct_change(fill_method=None)
mu = r.groupby(clean["ticker_clean"]).transform("mean")
sd = r.groupby(clean["ticker_clean"]).transform(lambda x: x.std(ddof=1))
clean["simple_return"] = r
clean["return_outlier_flag"] = ((r - mu).abs() > 3 * sd) & (sd > 0)

print("Spike bars repaired:", len(spike_idx), "| outliers flagged now:", clean["return_outlier_flag"].sum())
display(before)
display(clean.loc[spike_idx, ["ticker_clean", "ts"] + fix_cols + ["adj_close"]])

Spike bars repaired: 4 | outliers flagged now: 5174


,ticker_clean,ts,open,high,low,close,adj_close
232637,AXP,1972-12-14,4.01,4.01,4.01,4.01,1.35
356588,TRV,1976-10-28,9.22,9.22,9.22,9.22,3.21
8968,TRV,1978-12-20,8.03,8.03,8.03,8.03,2.80
142198,TRV,1981-08-10,12.06,12.06,12.06,12.06,4.20


,ticker_clean,ts,open,high,low,close,adj_close
232637,AXP,1972-12-14,4.595,4.595,4.595,4.595,1.546945
356588,TRV,1976-10-28,4.650,4.650,4.650,4.650,1.618926
8968,TRV,1978-12-20,4.055,4.055,4.055,4.055,1.413948
142198,TRV,1981-08-10,6.030,6.030,6.030,6.030,2.100000


In [28]:
px = ["open", "high", "low", "close", "adj_close"]
assert np.isfinite(clean[px]).all().all() and (clean[px] > 0).all().all()
assert (clean["low"] <= clean["open"]).all() and (clean["open"] <= clean["high"]).all()
assert (clean["low"] <= clean["close"]).all() and (clean["close"] <= clean["high"]).all()
assert (clean["adj_close"] <= clean["close"] * (1 + 1e-6)).all()
assert not clean.duplicated(["ticker_clean", "ts"]).any()
assert np.isfinite(clean["volume"]).all() and (clean["volume"] >= 0).all()
print("All validation checks passed. Rows:", len(clean))

All validation checks passed. Rows: 368246


In [29]:
out = (clean.rename(columns={"ticker_clean": "ticker"})
            [["ticker", "ts", "open", "high", "low", "close", "adj_close", "volume",
              "adj_close_repaired", "spike_repaired", "volume_repaired",
              "zero_volume_flag", "return_outlier_flag"]]
            .sort_values(["ticker", "ts"]).reset_index(drop=True))
out["ts"] = pd.to_datetime(out["ts"]).dt.strftime("%Y-%m-%d")
out.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)

print(pd.read_sql_query(
    "SELECT COUNT(*) AS n, COUNT(DISTINCT ticker) AS tickers, MIN(ts) AS first, MAX(ts) AS last, "
    "SUM(adj_close_repaired) AS adj_repaired, SUM(spike_repaired) AS spikes FROM ohlc_hw_cleaned", conn))

        n  tickers       first        last  adj_repaired  spikes
0  368246       30  1962-01-02  2024-12-06         28791       4


In [30]:
before_counts = flagged.groupby("ticker_clean").size()
after_counts = clean[clean["return_outlier_flag"]].groupby("ticker_clean").size()
display(after_counts.sub(before_counts, fill_value=0).sort_values(ascending=False).head(5))

ticker_clean
TRV     119
MCD       1
AMZN      0
BA        0
AAPL      0
dtype: int64

In [31]:
rng = de_duplicate.groupby("ticker_clean")["ts"].agg(first="min", last="max", observed="nunique")
rng["missing"] = gaps.groupby("ticker_clean").size().reindex(rng.index, fill_value=0)
for t, row in rng.iterrows():
    print(f"{t} is missing {row['missing']} trading days between {row['first']} and {row['last']} ({row['observed']} observed)")

AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18 (11050 observed)
AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18 (10421 observed)
AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06 (6937 observed)
AXP is missing 0 trading days between 1972-04-03 and 2024-10-18 (13252 observed)
BA is missing 0 trading days between 1962-01-02 and 2024-10-18 (15784 observed)
CAT is missing 0 trading days between 1962-01-02 and 2024-10-18 (15808 observed)
CRM is missing 0 trading days between 2004-06-23 and 2024-10-18 (5117 observed)
CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18 (8734 observed)
CVX is missing 0 trading days between 1962-01-02 and 2024-12-06 (15843 observed)
DIS is missing 0 trading days between 1962-01-02 and 2024-12-06 (15867 observed)
GS is missing 0 trading days between 1999-05-04 and 2024-10-18 (6408 observed)
HD is missing 0 trading days between 1981-09-22 and 2024-10-18 (10861 observed)
HON is missing 0 trading days b

In [32]:
dup_toy = pd.DataFrame({"ticker_clean": ["A", "A", "A", "B"],
                        "ts": ["2024-01-02"] * 4,
                        "close": [10.0, 10.0, 10.5, 20.0]})   # rows 0 and 1 identical; row 2 conflicts
dedup_toy = dup_toy.drop_duplicates()
conflicts = dedup_toy[dedup_toy.duplicated(["ticker_clean", "ts"], keep=False)]
print(len(dup_toy) - len(dedup_toy), "exact duplicate dropped;", len(conflicts), "rows share a key with different values:")
display(conflicts)

1 exact duplicate dropped; 2 rows share a key with different values:


,ticker_clean,ts,close
0,A,2024-01-02,10.0
2,A,2024-01-02,10.5
